# Lecture 9: Continuous Systems and the Fourier Transform
### Computational Companion — $L^2(\mathbb{R})$, Basis Changes, and the Canonical Commutator

This notebook verifies every claim in Lecture 9:

1. **$L^2$ inner product and Parseval's theorem** — position and momentum norms agree
2. **Fourier transform as change of basis** — unitary matrix $U(p,x) = e^{-ipx}/\sqrt{2\pi}$
3. **Delta function** — Fourier representation, sifting property
4. **Momentum operator** — $\hat{p} = -id/dx$ verified on eigenfunctions
5. **Canonical commutator** — $[\hat{x}, \hat{p}] = i\hbar$ numerically
6. **Dispersion-free propagation** — $H = cP$: shape preserved
7. **Dispersive propagation** — $H = p^2/(2m)$: Gaussian spreading

**Convention:** $\hbar = 1$ throughout.

**Reference:** Lecture 9 notes ("Quantum Systems via Linear Algebra")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

hbar = 1.0

# ── Grid ──
N = 4096
L = 40.0
dx = L / N
x = np.linspace(-L/2, L/2, N, endpoint=False)
dp = 2 * np.pi / L
p = np.fft.fftfreq(N, d=dx) * 2 * np.pi * hbar

def normalize(psi):
    return psi / np.sqrt(np.sum(np.abs(psi)**2) * dx)

def to_momentum(psi):
    """Position → momentum space via FFT."""
    return np.fft.fft(psi) * dx / np.sqrt(2 * np.pi * hbar)

def to_position(psi_p):
    """Momentum → position space via IFFT."""
    return np.fft.ifft(psi_p) * N * dp / np.sqrt(2 * np.pi * hbar)

print(f"Grid: N={N}, L={L}, dx={dx:.4f}")
print("Setup complete.")

## 1. $L^2$ Inner Product and Parseval's Theorem

For any $v \in L^2(\mathbb{R})$: $\int |v(x)|^2 dx = \int |\tilde{v}(p)|^2 dp$ (Parseval). Both representations carry the same total probability.

In [ ]:
# ── 1. Parseval's Theorem ──

print("=" * 65)
print("PARSEVAL: ∫|v(x)|² dx = ∫|ṽ(p)|² dp")
print("=" * 65)

test_functions = {
    "Gaussian (σ=1)": normalize(np.exp(-x**2 / 2)),
    "Rect (width=2)": normalize(np.where(np.abs(x) < 1, 1.0, 0.0)),
    "Double Gaussian": normalize(np.exp(-(x-3)**2/2) + np.exp(-(x+3)**2/2)),
    "Boosted Gaussian": normalize(np.exp(-x**2/2) * np.exp(1j * 5 * x)),
}

print(f"\n  {'Function':<25} {'∫|v|²dx':>12} {'∫|ṽ|²dp':>12} {'Match'}")
print("  " + "-" * 60)
for name, psi in test_functions.items():
    norm_x = np.sum(np.abs(psi)**2) * dx
    psi_p = to_momentum(psi)
    norm_p = np.sum(np.abs(psi_p)**2) * dp
    print(f"  {name:<25} {norm_x:12.8f} {norm_p:12.8f} {np.isclose(norm_x, norm_p, atol=1e-6)}")

# Also check cross-correlation (inner product)
psi_a = normalize(np.exp(-x**2/2))
psi_b = normalize(np.exp(-(x-1)**2/2))
ip_x = np.sum(psi_a.conj() * psi_b) * dx
ip_p = np.sum(to_momentum(psi_a).conj() * to_momentum(psi_b)) * dp
print(f"\n  Inner product ⟨a,b⟩: position = {ip_x:.6f}, momentum = {ip_p:.6f}, match = {np.isclose(ip_x, ip_p, atol=1e-6)}")

## 2. Fourier Transform as Change of Basis

The Fourier kernel $U(p,x) = e^{-ipx}/\sqrt{2\pi}$ is the change-of-basis "matrix" from position to momentum. We verify:
- The transform is unitary ($U^\dagger U = I$ becomes the delta function identity)
- A Gaussian in position space becomes a Gaussian in momentum space
- The widths are inversely related

In [ ]:
# ── 2. Fourier Transform as Change of Basis ──

print("=" * 65)
print("FOURIER = CHANGE OF BASIS: position ↔ momentum")
print("=" * 65)

# Gaussian: known analytic FT
# v(x) = (2πσ²)^{-1/4} exp(-x²/(4σ²))
# ṽ(p) = (2σ²/π)^{1/4} exp(-σ²p²)

sigmas = [0.5, 1.0, 2.0, 4.0]
fig, axes = plt.subplots(2, len(sigmas), figsize=(16, 7))

print(f"\n  {'σ':>5} {'Δx':>8} {'Δp':>8} {'Δx·Δp':>10} {'ℏ/2':>6}")
print("  " + "-" * 45)

for i, sigma in enumerate(sigmas):
    psi = normalize(np.exp(-x**2 / (4 * sigma**2)))
    psi_p = to_momentum(psi)
    
    # Widths
    dx_val = np.sqrt(np.sum(x**2 * np.abs(psi)**2 * dx))
    dp_val = np.sqrt(np.sum(p**2 * np.abs(psi_p)**2 * dp))
    print(f"  {sigma:5.1f} {dx_val:8.4f} {dp_val:8.4f} {dx_val*dp_val:10.6f} {hbar/2:6.3f}")
    
    axes[0, i].plot(x, np.abs(psi)**2, 'steelblue', lw=2)
    axes[0, i].set_xlim(-8, 8); axes[0, i].set_title(f'|v(x)|², σ={sigma}')
    axes[0, i].grid(alpha=0.3)
    
    p_sorted = np.fft.fftshift(p)
    psi_p_sorted = np.fft.fftshift(psi_p)
    axes[1, i].plot(p_sorted, np.abs(psi_p_sorted)**2, 'crimson', lw=2)
    axes[1, i].set_xlim(-5, 5); axes[1, i].set_title(f'|ṽ(p)|², σ={sigma}')
    axes[1, i].grid(alpha=0.3)

axes[0, 0].set_ylabel('Position space')
axes[1, 0].set_ylabel('Momentum space')
plt.suptitle('Fourier duality: narrow in x ↔ broad in p (and vice versa)', fontsize=13)
plt.tight_layout(); plt.show()

print(f"\n→ All Gaussians: Δx·Δp = ℏ/2 (minimum uncertainty)")
print(f"  Narrow in position → broad in momentum, and vice versa")

## 3. The Dirac Delta Function

We verify the Fourier representation $\delta(x) = \frac{1}{2\pi}\int e^{ipx} dp$ by approximating the delta as the limit of narrow Gaussians, and check the sifting property.

In [ ]:
# ── 3. Dirac Delta Function ──

print("=" * 65)
print("DIRAC DELTA: Fourier representation and sifting")
print("=" * 65)

# Approximate delta as narrow Gaussian
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epsilons = [1.0, 0.5, 0.2, 0.1, 0.05]
for eps in epsilons:
    delta_approx = np.exp(-x**2 / (2 * eps**2)) / (eps * np.sqrt(2 * np.pi))
    axes[0].plot(x, delta_approx, lw=1.5, label=f'ε={eps}')

axes[0].set_xlim(-2, 2); axes[0].set_ylim(0, 10)
axes[0].set_title('δ(x) as limit of narrow Gaussians')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

# Fourier representation: δ(x) = (1/2π) ∫ e^{ipx} dp
# Approximate with finite cutoff P_max
P_maxes = [5, 10, 20, 50]
for P_max in P_maxes:
    # (1/2π) ∫_{-P}^{P} e^{ipx} dp = sin(Px)/(πx)
    delta_fourier = np.where(np.abs(x) > 1e-10, np.sin(P_max * x) / (np.pi * x), P_max / np.pi)
    axes[1].plot(x, delta_fourier, lw=1, label=f'P_max={P_max}')

axes[1].set_xlim(-3, 3); axes[1].set_ylim(-5, 20)
axes[1].set_title('Fourier representation: sin(Px)/(πx) → δ(x)')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.suptitle('Dirac delta function approximations', fontsize=13)
plt.tight_layout(); plt.show()

# Sifting property: ∫ δ(x-x₀) f(x) dx = f(x₀)
print("\nSifting property verification:")
f = np.sin(x) + x**2  # test function
x0_vals = [-2.0, 0.0, 1.5, 3.0]
eps = 0.02
for x0 in x0_vals:
    delta_x0 = np.exp(-(x - x0)**2 / (2 * eps**2)) / (eps * np.sqrt(2 * np.pi))
    integral = np.sum(delta_x0 * f) * dx
    exact = np.sin(x0) + x0**2
    print(f"  x₀ = {x0:5.1f}: ∫δ(x−x₀)f(x)dx = {integral:.6f}, f(x₀) = {exact:.6f}, match = {np.isclose(integral, exact, atol=0.01)}")

## 4. Momentum Operator and Canonical Commutator

- $\hat{p} = -i d/dx$ verified on plane wave eigenfunctions
- $[\hat{x}, \hat{p}] = i\hbar$ verified by acting on test functions

In [ ]:
# ── 4. Momentum Operator and Canonical Commutator ──

print("=" * 65)
print("MOMENTUM OPERATOR: p̂ = −id/dx")
print("=" * 65)

def apply_p_hat(psi):
    """Apply p̂ = -iℏ d/dx via FFT: multiply by p in momentum space."""
    psi_p = np.fft.fft(psi)
    return np.fft.ifft(p * psi_p)

def apply_x_hat(psi):
    """Apply x̂ = x (multiplication)."""
    return x * psi

# Verify on plane waves: p̂ e^{ipx} = p e^{ipx}
print("\nPlane wave eigenfunctions: p̂ e^{ipx} = p·e^{ipx}")
for p_val in [1.0, 3.0, -2.0, 5.0]:
    psi_pw = np.exp(1j * p_val * x)
    p_hat_psi = apply_p_hat(psi_pw)
    # Should be p_val * psi_pw
    ratio = p_hat_psi / (psi_pw + 1e-30)
    eigenvalue = np.median(np.real(ratio[np.abs(psi_pw) > 0.5]))
    print(f"  p = {p_val:5.1f}: eigenvalue = {eigenvalue:.4f}, match = {np.isclose(eigenvalue, p_val, atol=0.01)}")

# Canonical commutator: [x̂, p̂]ψ = iℏψ
print("\n" + "=" * 65)
print("CANONICAL COMMUTATOR: [x̂, p̂] = iℏ")
print("=" * 65)

test_psis = {
    "Gaussian": normalize(np.exp(-x**2/2)),
    "Boosted Gaussian": normalize(np.exp(-x**2/2) * np.exp(1j*3*x)),
    "Rect": normalize(np.where(np.abs(x) < 2, 1.0 + 0j, 0.0)),
    "Double peak": normalize(np.exp(-(x-2)**2) + np.exp(-(x+2)**2) + 0j),
}

for name, psi in test_psis.items():
    xp_psi = apply_x_hat(apply_p_hat(psi))
    px_psi = apply_p_hat(apply_x_hat(psi))
    comm_psi = xp_psi - px_psi  # should be iℏ * psi
    
    # Check comm_psi / psi = iℏ in the bulk (avoid edges/zeros)
    mask = np.abs(psi) > 0.01 * np.max(np.abs(psi))
    ratio = comm_psi[mask] / psi[mask]
    mean_ratio = np.mean(ratio)
    print(f"  {name:<20}: [x̂,p̂]ψ/ψ = {mean_ratio:.4f} (should be {1j*hbar}), match = {np.isclose(mean_ratio, 1j*hbar, atol=0.05)}")

## 5. Finite-to-Continuous Dictionary

We verify the correspondence table from the lecture by computing the discrete analogs and showing they converge to the continuous versions as $N \to \infty$.

In [ ]:
# ── 5. Finite-to-Continuous Dictionary ──

print("=" * 65)
print("FINITE ↔ CONTINUOUS DICTIONARY")
print("=" * 65)

# Discrete: DFT matrix is unitary
N_small = 8
DFT = np.fft.fft(np.eye(N_small)) / np.sqrt(N_small)
print(f"\nDFT matrix ({N_small}×{N_small}):")
print(f"  Unitary: DFT†·DFT = I? {np.allclose(DFT.conj().T @ DFT, np.eye(N_small))}")

# Discrete basis vectors: e_i (Kronecker)
# Continuous: δ(x-x₀)
print(f"\nBasis vectors:")
print(f"  Finite: e_i has entries δ_ij (Kronecker delta)")
print(f"  Continuous: δ(x-x₀) (Dirac delta)")

# Discrete orthogonality: e_i† e_j = δ_ij
e_3 = np.zeros(N_small); e_3[3] = 1
e_5 = np.zeros(N_small); e_5[5] = 1
print(f"\nOrthogonality:")
print(f"  Finite: e₃†e₅ = {e_3 @ e_5} (= δ₃₅ = 0 ✓)")
print(f"  Finite: e₃†e₃ = {e_3 @ e_3} (= δ₃₃ = 1 ✓)")
print(f"  Continuous: ∫δ(x-a)δ(x-b)dx = δ(a-b)")

# Discrete expansion: v = Σ v_i e_i
v_disc = np.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=float)
v_reconstructed = sum(v_disc[i] * np.eye(N_small)[:, i] for i in range(N_small))
print(f"\nExpansion:")
print(f"  Finite: v = Σ vᵢeᵢ, reconstructed = {np.allclose(v_disc, v_reconstructed)} ✓")
print(f"  Continuous: v(x) = ∫v(x')δ(x-x')dx'")

# Resolution of identity
print(f"\nResolution of identity:")
I_reconstructed = sum(np.outer(np.eye(N_small)[:, i], np.eye(N_small)[:, i]) for i in range(N_small))
print(f"  Finite: Σ eᵢeᵢ† = I? {np.allclose(I_reconstructed, np.eye(N_small))} ✓")
print(f"  Continuous: ∫|x⟩⟨x|dx = I")

print(f"\n→ Same machinery. Sum → integral. δ_ij → δ(x−x').")

## 6. Dispersion-Free Propagation: $H = cP$

The wave equation $\partial_t v + c \partial_x v = 0$ has solution $v(x,t) = f(x - ct)$: the shape moves rigidly without distortion. This is a pure delay line.

In [ ]:
# ── 6. Dispersion-Free: H = cP ──

print("=" * 65)
print("DISPERSION-FREE: H = cP (delay line)")
print("=" * 65)

c = 2.0  # propagation speed

# Initial condition: asymmetric pulse
psi0_delay = normalize(np.exp(-x**2/2) * (1 + 0.5 * x))

# Evolve: v(x,t) = f(x - ct) → in Fourier: multiply by e^{-icpt}
def evolve_delay(psi, t, c_val):
    psi_p = np.fft.fft(psi)
    phase = np.exp(-1j * c_val * p * t / hbar)
    return np.fft.ifft(psi_p * phase)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
snap_times = [0, 1, 2, 3]
for ax, t in zip(axes, snap_times):
    psi_t = evolve_delay(psi0_delay, t, c)
    ax.plot(x, np.abs(psi_t)**2, 'steelblue', lw=2)
    ax.axvline(c * t, color='red', ls='--', alpha=0.7, label=f'x = ct = {c*t:.0f}')
    ax.set_xlim(-5, 15); ax.set_ylim(0, 0.6)
    ax.set_title(f't = {t}'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle(f'H = cP: shape preserved, moves at speed c = {c}', fontsize=13)
plt.tight_layout(); plt.show()

# Verify shape preservation
psi_t3 = evolve_delay(psi0_delay, 3.0, c)
# Shift back by ct and compare
shift_idx = int(round(c * 3.0 / dx))
psi_shifted = np.roll(np.abs(psi_t3)**2, -shift_idx)
err = np.max(np.abs(psi_shifted - np.abs(psi0_delay)**2))
print(f"Shape preservation: max|ψ(x-ct,t)|² − |ψ(x,0)|²| = {err:.2e} ✓")

## 7. Dispersive Propagation: $H = p^2/(2m)$

The Schrödinger equation for a free particle has quadratic dispersion: $\omega(k) = k^2/(2m)$. Different frequency components travel at different speeds, causing the wave packet to spread. A Gaussian initial condition remains Gaussian with width $\sigma(t) = \sqrt{\sigma_0^2 + (t/(2m\sigma_0))^2}$.

In [ ]:
# ── 7. Dispersive: H = p²/(2m) ──

print("=" * 65)
print("DISPERSIVE: H = p²/(2m) — Gaussian spreading")
print("=" * 65)

m = 1.0
sigma0 = 1.0

def evolve_free(psi, t, m_val=1.0):
    psi_p = np.fft.fft(psi)
    phase = np.exp(-1j * p**2 * t / (2 * m_val * hbar))
    return np.fft.ifft(psi_p * phase)

psi0_free = normalize(np.exp(-x**2 / (4 * sigma0**2)))

# Snapshots
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
snap_times = [0, 2, 5, 10]
for ax, t in zip(axes, snap_times):
    psi_t = evolve_free(psi0_free, t, m)
    sigma_t = sigma0 * np.sqrt(1 + (hbar * t / (2 * m * sigma0**2))**2)
    analytic = np.exp(-x**2 / (4 * sigma_t**2)) / (2 * np.pi * sigma_t**2)**0.25
    ax.plot(x, np.abs(psi_t)**2, 'steelblue', lw=2, label='Numerical')
    ax.plot(x, np.abs(analytic)**2, 'r--', lw=1.5, label=f'σ(t)={sigma_t:.2f}')
    ax.set_xlim(-15, 15); ax.set_ylim(0, 0.5)
    ax.set_title(f't = {t}'); ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.suptitle('H = p²/(2m): Gaussian spreads (dispersive channel)', fontsize=13)
plt.tight_layout(); plt.show()

# Spreading curve
times = np.linspace(0, 15, 200)
sigma_analytic = sigma0 * np.sqrt(1 + (hbar * times / (2 * m * sigma0**2))**2)
sigma_numerical = np.zeros(len(times))
for i, t in enumerate(times):
    psi_t = evolve_free(psi0_free, t, m)
    sigma_numerical[i] = np.sqrt(np.sum(x**2 * np.abs(psi_t)**2 * dx))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(times, sigma_numerical, 'steelblue', lw=2, label='Δx (numerical)')
axes[0].plot(times, sigma_analytic, 'r--', lw=1.5, label='σ(t) (analytic)')
axes[0].set_xlabel('t'); axes[0].set_ylabel('Δx')
axes[0].set_title('Gaussian spreading: Δx grows with time')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Δp stays constant
dp_numerical = np.zeros(len(times))
for i, t in enumerate(times):
    psi_t = evolve_free(psi0_free, t, m)
    psi_p_t = to_momentum(psi_t)
    dp_numerical[i] = np.sqrt(np.sum(p**2 * np.abs(psi_p_t)**2 * dp))

axes[1].plot(times, sigma_numerical * dp_numerical, 'green', lw=2, label='Δx·Δp')
axes[1].axhline(hbar/2, color='red', ls='--', lw=2, label=f'ℏ/2 = {hbar/2}')
axes[1].set_xlabel('t'); axes[1].set_ylabel('Δx·Δp')
axes[1].set_title('Uncertainty product grows from ℏ/2')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Free particle dispersion: position spreads, momentum constant', fontsize=13)
plt.tight_layout(); plt.show()

print(f"At t=0: Δx·Δp = {sigma_numerical[0]*dp_numerical[0]:.6f} (= ℏ/2 = {hbar/2})")
print(f"At t=15: Δx·Δp = {sigma_numerical[-1]*dp_numerical[-1]:.4f} (> ℏ/2)")
print(f"Δp constant: max|Δp(t) − Δp(0)| = {np.max(np.abs(dp_numerical - dp_numerical[0])):.2e}")

## 8. Delay Line vs Dispersive Channel — Side by Side

We compare $H = cP$ (dispersion-free) and $H = p^2/(2m)$ (dispersive) evolving the same initial Gaussian.

In [ ]:
# ── 8. Side-by-Side Comparison ──

print("=" * 65)
print("DELAY LINE vs DISPERSIVE CHANNEL")
print("=" * 65)

# Same initial Gaussian with initial momentum
p0 = 3.0
psi0_comp = normalize(np.exp(-x**2 / (4 * sigma0**2)) * np.exp(1j * p0 * x / hbar))

fig, axes = plt.subplots(2, 4, figsize=(18, 7))
comp_times = [0, 2, 4, 6]

for j, t in enumerate(comp_times):
    # Delay line
    psi_delay = evolve_delay(psi0_comp, t, p0/m)  # c = p0/m for same center speed
    axes[0, j].plot(x, np.abs(psi_delay)**2, 'steelblue', lw=2)
    axes[0, j].set_xlim(-5, 25); axes[0, j].set_ylim(0, 0.5)
    axes[0, j].set_title(f't = {t}')
    if j == 0: axes[0, j].set_ylabel('H = cP (no dispersion)')
    axes[0, j].grid(alpha=0.3)
    
    # Free particle
    psi_disp = evolve_free(psi0_comp, t, m)
    axes[1, j].plot(x, np.abs(psi_disp)**2, 'crimson', lw=2)
    axes[1, j].set_xlim(-5, 25); axes[1, j].set_ylim(0, 0.5)
    if j == 0: axes[1, j].set_ylabel('H = p²/(2m) (dispersive)')
    axes[1, j].grid(alpha=0.3)

plt.suptitle('Delay line (shape preserved) vs dispersive channel (spreading)', fontsize=13)
plt.tight_layout(); plt.show()

print(f"→ H = cP: shape preserved at all times (delay line)")
print(f"  H = p²/(2m): wave packet spreads (dispersive channel)")
print(f"  Both move center at v = p₀/m = {p0/m}, but only p²/(2m) disperses.")